# Experiment 001: Dense Baseline (mDPR)

**Date:** January 16, 2026  
**Objective:** Establish mDPR baseline with Identity query enhancement (no enhancement)  
**Expected Results:**
- Recall@100: ~0.841
- NDCG@10: ~0.499
- Recall@10: (thesis baseline)

**Query Enhancement:** Identity (returns query unchanged)

## Setup

### Step 1: Clone Repository and Install Dependencies

In [ ]:
# Clone repository
!git clone https://github.com/Osmanoor/graduation.git
%cd graduation/arabic-rag-query-enhancement

# Install Java 21
!apt-get install -qq openjdk-21-jdk-headless

# Install Python dependencies
!pip install -q pyserini faiss-cpu pytrec-eval transformers torch

print("\n" + "="*60)
print("✓ Installation complete")
print("="*60)
print("⚠️ IMPORTANT: Restart runtime now!")
print("   1. Click 'Runtime' → 'Restart runtime'")
print("   2. Then run cells starting from 'Step 2' below")
print("="*60)

### Step 2: Configure Environment (Run After Restart)

In [ ]:
# Navigate to project directory
%cd /content/graduation/arabic-rag-query-enhancement

# Configure environment
import os
import sys

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Add src to path
sys.path.insert(0, '/content/graduation/arabic-rag-query-enhancement')

# Verify Java
!java -version

print("\n✓ Environment configured")
print("✓ Ready to run experiment")

## Import Modules

In [ ]:
from src.utils.data_loader import MIRACLDataLoader
from src.retrievers.dense import mDPRRetriever
from src.enhancers.base import IdentityEnhancer
from src.evaluation.metrics import RetrievalEvaluator, save_results, save_metrics, print_metrics

import torch
from tqdm.notebook import tqdm

print("✓ Modules imported")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Load Data

In [ ]:
# Load MIRACL Arabic dev set
data_loader = MIRACLDataLoader(language="ar", split="dev")
topics, qrels = data_loader.load_all()

print(f"\nDataset Statistics:")
print(f"  Queries: {len(topics)}")
print(f"  Qrels: {len(qrels)}")

# Show sample
sample_qid = list(topics.keys())[0]
print(f"\nSample Query:")
print(f"  ID: {sample_qid}")
print(f"  Text: {topics[sample_qid]['title']}")
print(f"  Relevant docs: {len(qrels.get(sample_qid, {}))}")

## Initialize Components

In [ ]:
# Initialize query enhancer (Identity = no enhancement)
enhancer = IdentityEnhancer()
print("✓ Query enhancer: Identity (baseline)")

# Initialize retriever
retriever = mDPRRetriever(
    index_name="miracl-v1.0-ar-mdpr-tied-pft-msmarco",
    encoder_name="castorini/mdpr-tied-pft-msmarco",
    device="cuda" if torch.cuda.is_available() else "cpu",
    batch_size=64
)

# Initialize evaluator
evaluator = RetrievalEvaluator(qrels)
print("✓ Evaluator initialized")

## Run Experiment

In [ ]:
print("="*60)
print("EXPERIMENT 001: Dense Baseline (Identity Enhancement)")
print("="*60)

# Prepare queries
query_ids = list(topics.keys())
query_texts = [topics[qid]['title'] for qid in query_ids]

# Apply query enhancement (Identity = no change)
print(f"\nApplying query enhancement...")
enhanced_queries = enhancer.enhance_batch(query_texts, query_ids)
print(f"✓ Enhanced {len(enhanced_queries)} queries")

# Verify enhancement (should be identical for Identity)
print(f"\nVerification:")
print(f"  Original: {query_texts[0]}")
print(f"  Enhanced: {enhanced_queries[0]}")
print(f"  Same: {query_texts[0] == enhanced_queries[0]}")

In [ ]:
# Run retrieval
print(f"\nRunning retrieval for {len(enhanced_queries)} queries...")
print("This will take ~2-3 minutes with T4 GPU")

search_results = retriever.search(enhanced_queries, k=100, show_progress=True)

print(f"✓ Retrieval complete")

In [ ]:
# Format results for evaluation
print("\nFormatting results...")
results = {}

for i, qid in enumerate(tqdm(query_ids, desc="Processing")):
    results[qid] = {}
    for docid, score in search_results[i]:
        results[qid][docid] = score

print(f"✓ Formatted {len(results)} query results")

## Evaluate Results

In [ ]:
# Compute metrics
print("\nEvaluating...")
metrics = evaluator.evaluate(results)

# Print results
print_metrics(metrics, "EXPERIMENT 001: Dense Baseline Results")

# Compare with expected
print("\nComparison with MIRACL Paper:")
print(f"  Recall@100: {metrics['recall_100']:.4f} (Expected: ~0.841)")
print(f"  NDCG@10:    {metrics['ndcg_cut_10']:.4f} (Expected: ~0.499)")

# Calculate achievement
recall100_achievement = (metrics['recall_100'] / 0.841) * 100
ndcg10_achievement = (metrics['ndcg_cut_10'] / 0.499) * 100

print(f"\nAchievement:")
print(f"  Recall@100: {recall100_achievement:.2f}%")
print(f"  NDCG@10:    {ndcg10_achievement:.2f}%")

## Save Results

In [ ]:
import os

# Create output directory
output_dir = "results/baseline_dense"
os.makedirs(output_dir, exist_ok=True)

# Save results in TREC format
save_results(
    results,
    f"{output_dir}/exp_001_baseline_dense.txt",
    run_name="exp_001_identity"
)

# Save metrics
save_metrics(
    metrics,
    f"{output_dir}/exp_001_metrics.json"
)

print("\n✓ All results saved")

## Summary

**Experiment:** 001 - Dense Baseline (Identity Enhancement)  
**Status:** Complete  

**Results:**
- Recall@10: [Will be filled after run]
- Recall@100: [Will be filled after run]
- NDCG@10: [Will be filled after run]
- MRR: [Will be filled after run]

**Next Steps:**
1. Document results in `experiments/exp_001_baseline_dense.md`
2. Analyze error patterns
3. Select first query enhancement technique
4. Run Experiment 002 with actual enhancement